In [113]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [114]:
import pandas as pd
import glob

path = '/content/drive/MyDrive/Dataset Arkavidia/ISPU/'
files = glob.glob(path + '*ispu*.csv')
simpan = []


for x in files:
  data = pd.read_csv(x, sep=None, engine='python')
  data.columns = data.columns.str.lower().str.strip()

  if 'tanggal' in data.columns and data['tanggal'].astype(str).str.len().max() <= 2:
        tahun_str = data['periode_data'].astype(str).str[:4]
        bulan_str = data['periode_data'].astype(str).str[4:6]
        hari_str = data['tanggal'].astype(str).str.zfill(2)

        data['tanggal'] = tahun_str + "-" + bulan_str + "-" + hari_str

  mapping = {
        'parameter_pencemar_kritis': 'critical',
        'parameter_kritis': 'critical',
        'pencemar_kritis': 'critical',
        'kategori': 'category',
        'categori': 'category',

        'pm_sepuluh': 'pm10', 'pm_10': 'pm10',
        'pm_duakomalima': 'pm25', 'pm_25': 'pm25',
        'lokasi_spku': 'stasiun',
        'sulfur_dioksida': 'so2', 'karbon_monoksida': 'co',
        'ozon': 'o3', 'nitrogen_dioksida': 'no2'
    }

  data.rename(columns=mapping, inplace=True)
  kolom_target = ['periode_data', 'tanggal', 'stasiun', 'pm10', 'pm25',
                    'so2', 'co', 'o3', 'no2', 'max', 'critical', 'category']

  ada_kolom = [y for y in kolom_target if y in data.columns]
  data = data[ada_kolom]
  simpan.append(data)



DataISPU = pd.concat(simpan,axis=0,ignore_index=True)
DataISPU['tanggal'] = pd.to_datetime(DataISPU['tanggal'],errors='coerce')
DataISPU = DataISPU[DataISPU['tanggal'].dt.year >= 2010]
DataISPU = DataISPU.dropna(subset=['tanggal'])
DataISPU = DataISPU.sort_values(by='tanggal').reset_index(drop=True)
print(f'Selesai GnG',{len(files)})
print(f"Rentang Data Akhir: {DataISPU['tanggal'].min()} sampai {DataISPU['tanggal'].max()}")


Selesai GnG {16}
Rentang Data Akhir: 2010-01-01 00:00:00 sampai 2025-08-31 00:00:00


/tmp/ipykernel_568/641586978.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  DataISPU['tanggal'] = pd.to_datetime(DataISPU['tanggal'],errors='coerce')


**EDA**

In [115]:
DataISPU.head()

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
0,201001,2010-01-01,DKI2 (Kelapa Gading),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
1,201001,2010-01-01,DKI3 (Jagakarsa),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
2,201001,2010-01-01,DKI5 (Kebon Jeruk),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
3,201001,2010-01-01,DKI1 (Bunderan HI),60,NaN,4,73,27,14,73,CO,SEDANG
4,201001,2010-01-01,DKI4 (Lubang Buaya),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA


In [116]:
DataISPU.tail()

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
16896,202508,2025-08-31,DKI4 Lubang Buaya,47.0,59.0,27.0,10.0,18.0,17.0,59.0,PM25,SEDANG
16897,202508,2025-08-31,DKI1 Bundaran Hotel Indonesia HI,42.0,70.0,29.0,12.0,15.0,24.0,70.0,PM25,SEDANG
16898,202508,2025-08-31,DKI2 Kelapa Gading,NaN,72.0,45.0,16.0,21.0,16.0,72.0,PM25,SEDANG
16899,202508,2025-08-31,DKI5 Kebon Jeruk,37.0,65.0,25.0,9.0,21.0,34.0,65.0,PM25,SEDANG
16900,202508,2025-08-31,DKI3 Jagakarsa,28.0,60.0,53.0,8.0,19.0,39.0,60.0,PM25,SEDANG


In [117]:
DataISPU.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16901 entries, 0 to 16900
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   periode_data  16901 non-null  int64         
 1   tanggal       16901 non-null  datetime64[ns]
 2   stasiun       16901 non-null  object        
 3   pm10          16715 non-null  object        
 4   pm25          6944 non-null   object        
 5   so2           16848 non-null  object        
 6   co            16857 non-null  object        
 7   o3            16858 non-null  object        
 8   no2           16835 non-null  object        
 9   max           16894 non-null  object        
 10  critical      15410 non-null  object        
 11  category      16900 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 1.5+ MB


In [118]:
len(DataISPU)

16901

**Data Cleaning**

In [119]:
polutan = [
    'pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'
]


for i in polutan:
  if i in DataISPU.columns:
    DataISPU[i] = DataISPU[i].astype(str).str.strip()
    DataISPU[i] = pd.to_numeric(DataISPU[i], errors = 'coerce')

In [120]:
DataISPU.head(200)

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
0,201001,2010-01-01,DKI2 (Kelapa Gading),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
1,201001,2010-01-01,DKI3 (Jagakarsa),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
2,201001,2010-01-01,DKI5 (Kebon Jeruk),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
3,201001,2010-01-01,DKI1 (Bunderan HI),60.0,NaN,4.0,73.0,27.0,14.0,73.0,CO,SEDANG
4,201001,2010-01-01,DKI4 (Lubang Buaya),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
...,...,...,...,...,...,...,...,...,...,...,...,...
195,201002,2010-02-09,DKI5 (Kebon Jeruk),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
196,201002,2010-02-09,DKI1 (Bunderan HI),52.0,NaN,7.0,30.0,34.0,16.0,52.0,PM10,SEDANG
197,201002,2010-02-09,DKI2 (Kelapa Gading),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA
198,201002,2010-02-09,DKI4 (Lubang Buaya),NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,TIDAK ADA DATA


In [121]:
import numpy as np
DataISPU['max'] = DataISPU['max'].replace(0, np.nan)

In [122]:
DataISPU.isnull().sum()

,0
periode_data,0
tanggal,0
stasiun,0
pm10,2167
pm25,10290
so2,1772
co,1667
o3,1803
no2,1770
max,1459


In [123]:
DataISPU['category'].value_counts()

,count
category,
SEDANG,10447
TIDAK SEHAT,2421
BAIK,2342
TIDAK ADA DATA,1458
SANGAT TIDAK SEHAT,200
O3,31
BERBAHAYA,1


In [124]:
DataISPU['category'] = DataISPU['category'].replace('TIDAK ADA DATA',np.nan)

In [125]:
DataISPU['category'].isnull().sum()

np.int64(1459)

In [126]:
from sklearn.impute import SimpleImputer

Imputer = SimpleImputer(strategy = 'most_frequent')

DataISPU['category'] = Imputer.fit_transform(DataISPU[['category']]).ravel()

In [127]:
hapus = ['SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT','BAIK']

DataISPU = DataISPU[~DataISPU['stasiun'].isin(hapus)]

In [128]:
DataISPU.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16870 entries, 0 to 16900
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   periode_data  16870 non-null  int64         
 1   tanggal       16870 non-null  datetime64[ns]
 2   stasiun       16870 non-null  object        
 3   pm10          14703 non-null  float64       
 4   pm25          6611 non-null   float64       
 5   so2           15111 non-null  float64       
 6   co            15204 non-null  float64       
 7   o3            15067 non-null  float64       
 8   no2           15100 non-null  float64       
 9   max           15411 non-null  float64       
 10  critical      15379 non-null  object        
 11  category      16870 non-null  object        
dtypes: datetime64[ns](1), float64(7), int64(1), object(3)
memory usage: 1.7+ MB


In [129]:
DataISPU['critical'].value_counts()

,count
critical,
PM25,5574
O3,5471
PM10,3054
SO2,500
"PM2,5",332
CO,260
2,112
1,30
NO2,22


In [130]:
valueCiritical = ['PM25', 'O3', 'PM10','SO2','PM2,5','CO','NO2']

DataISPU = DataISPU[DataISPU['critical'].isin(valueCiritical)]

In [131]:
DataISPU.isnull().sum()

,0
periode_data,0
tanggal,0
stasiun,0
pm10,692
pm25,8769
so2,301
co,206
o3,344
no2,310
max,0


In [132]:
DataISPU.head()

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
3,201001,2010-01-01,DKI1 (Bunderan HI),60.0,NaN,4.0,73.0,27.0,14.0,73.0,CO,SEDANG
9,201001,2010-01-02,DKI1 (Bunderan HI),32.0,NaN,2.0,16.0,33.0,9.0,33.0,O3,BAIK
12,201001,2010-01-03,DKI1 (Bunderan HI),27.0,NaN,2.0,19.0,20.0,9.0,27.0,PM10,BAIK
18,201001,2010-01-04,DKI1 (Bunderan HI),22.0,NaN,2.0,16.0,15.0,6.0,22.0,PM10,BAIK
22,201001,2010-01-05,DKI1 (Bunderan HI),25.0,NaN,2.0,17.0,15.0,8.0,25.0,PM10,BAIK


In [133]:
len(DataISPU)

15213

In [134]:
DataISPU['category'].value_counts()

,count
category,
SEDANG,10297
TIDAK SEHAT,2415
BAIK,2301
SANGAT TIDAK SEHAT,199
BERBAHAYA,1


In [141]:
label = {
    'SANGAT TIDAK SEHAT' : 0,
    'TIDAK SEHAT' : 1,
    'SEDANG' : 2,
    'BAIK' : 3,
    'BERBAHAYA' : 4
}



In [136]:
DataISPU.isnull().sum()

,0
periode_data,0
tanggal,0
stasiun,0
pm10,692
pm25,8769
so2,301
co,206
o3,344
no2,310
max,0


In [142]:
DataISPU['target_category'] = DataISPU['category'].map(label)

**PREPROCESSING PIPELINE**

In [157]:
from sklearn.model_selection import train_test_split

features = ['tanggal', 'stasiun', 'pm10', 'pm25','so2',
            'co', 'o3', 'no2', 'max','critical']

X = DataISPU[features]
y = DataISPU['target_category']



TrainX,ValX,TrainY,ValY = train_test_split(X,y,test_size=0.2,
                                           random_state =42)


numeric_features = TrainX.select_dtypes(include=['int64','float64']).columns
categorical_features= TrainX.select_dtypes(include = ['object']).columns

In [158]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


numeric_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


categorical_transform = Pipeline(steps=[
    ('OneHot', OneHotEncoder(handle_unknown = 'ignore',
                             sparse_output = False))
])

In [163]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transform, numeric_features),
        ('cat', categorical_transform,categorical_features)
    ]
)

TrainXNew = preprocessor.fit_transform(TrainX)
ValXNew = preprocessor.transform(ValX)

In [164]:
np.isnan(TrainXNew).sum()

np.int64(0)